# Notebook 0: the skeleton of every machine-learning notebook

Before we touch the plant data, here is the whole recipe on a small, simple problem: **predict tomorrow's water demand of a town from the weather.** The data is made up inside the notebook, so nothing needs downloading.

Every ML notebook you will meet, including the project's, has these same steps in this order:

| Step | What it does | Question it answers |
|---|---|---|
| 1. Collect | read the raw file, then look: shape, types, summary | where does the data come from, what is in it? |
| 2. Clean | duplicates, wrong types, gaps, impossible values | what is wrong with it? |
| 3. Inputs and target | choose X (what we know) and y (what we want) | what goes in, what comes out? |
| 4. Split | train / validation / test, by time | which data may the model learn from? |
| 5. Scale | standardise with the training numbers only | are all inputs on a similar scale? |
| 6. Feed | tensors and batches | how does the data get into the model? |
| 7. Model | the function with knobs | what shape of function? |
| 8. Train | loss, gradients, update, check validation | how are the knobs set? |
| 9. Size | count the knobs, compare with the examples | is the model too big for the data? |
| 10. Settings | try a few sizes and learning rates on validation | how big, how fast? |
| 11. Test | once, in real units, against a simple baseline | is it actually good? |
| 12. Save | weights + scaler + settings | can someone else use it? |

Run the cells from top to bottom with **Shift + Enter**.

In [ ]:
# Step 0: the libraries. Every notebook starts with its imports.
import numpy as np
import pandas as pd
import torch
from torch import nn

torch.manual_seed(0)       # fixed seeds: the same numbers every time you run
rng = np.random.default_rng(0)

## 1. Collect

In real work this is a file you were given, an export from a database, or a download. Here we write a small CSV first, with a few typical problems hidden in it, and then read it back as if someone had sent it to us.

In [ ]:
# Make a fake "daily_demand.csv": 3 years of daily data.
days = pd.date_range("2021-01-01", periods=3 * 365, freq="D")
t = np.arange(len(days))
temperature = 25 + 6 * np.sin(2 * np.pi * (t - 100) / 365) + rng.normal(0, 1.5, len(t))   # °C
rainfall = np.where(rng.random(len(t)) < 0.25, rng.gamma(2, 6, len(t)), 0.0)                  # mm
weekend = (days.dayofweek >= 5).astype(int)
demand = 900 + 18 * temperature - 2.5 * rainfall - 60 * weekend + 0.15 * t + rng.normal(0, 25, len(t))  # thousand litres

raw = pd.DataFrame({"date": days.strftime("%Y-%m-%d"), "temperature": temperature.round(1),
                    "rainfall": rainfall.round(1), "demand": demand.round(0)})
raw.loc[40, "temperature"] = np.nan            # a missing reading
raw.loc[300, "demand"] = raw.loc[300, "demand"] * 10   # a typing error (an extra zero)
raw.loc[500, "rainfall"] = -3.0                # an impossible value
raw = pd.concat([raw, raw.iloc[[700]]])        # a duplicated row
raw.to_csv("daily_demand.csv", index=False)

# This is the line you would normally start with:
df = pd.read_csv("daily_demand.csv")

### Then look, before you do anything else

In [ ]:
print(df.shape)          # rows, columns
print(df.dtypes)         # is every column the type you expect?
df.head()

In [ ]:
df.describe().round(1)   # look at min and max: anything impossible?

Already visible: `demand` has a maximum far above everything else, and `rainfall` has a negative minimum. Rain cannot be negative.

## 2. Clean

Fix only what you can explain, and write down every fix.

In [ ]:
df["date"] = pd.to_datetime(df["date"])                  # text → real dates
print("duplicate dates:", df["date"].duplicated().sum())
df = df.drop_duplicates("date").sort_values("date").set_index("date")

print("missing values:\n", df.isna().sum())
df["temperature"] = df["temperature"].interpolate(limit=2)  # fill short gaps only

df.loc[df["rainfall"] < 0, "rainfall"] = np.nan             # impossible → unknown
df["rainfall"] = df["rainfall"].fillna(0)

median = df["demand"].rolling(15, center=True, min_periods=1).median()
spike = (df["demand"] - median).abs() > 300                 # far from its neighbours
print("spikes found:", int(spike.sum()))
df.loc[spike, "demand"] = median[spike]                     # replace by the local median

df.describe().round(1)

## 3. Inputs (X) and target (y)

**Target y:** tomorrow's demand. **Inputs X:** only what we know today: tomorrow's weather forecast, whether tomorrow is a weekend, and today's demand. Never put anything in X that you would not have at the time of the prediction.

In [ ]:
data = pd.DataFrame({
    "temp_tomorrow": df["temperature"].shift(-1),
    "rain_tomorrow": df["rainfall"].shift(-1),
    "weekend_tomorrow": ((df.index + pd.Timedelta(days=1)).dayofweek >= 5).astype(int),
    "demand_today": df["demand"],
    "y_demand_tomorrow": df["demand"].shift(-1),
}).dropna()

X = data.drop(columns="y_demand_tomorrow").to_numpy(dtype=np.float32)
y = data["y_demand_tomorrow"].to_numpy(dtype=np.float32).reshape(-1, 1)
print("X:", X.shape, " y:", y.shape)     # (examples, inputs) and (examples, outputs)

## 4. Split: train, validation, test, by time

- **train**: the model learns from it
- **validation**: we use it to make choices (when to stop, how big a model)
- **test**: looked at **once**, at the very end

The data is a time series, so we split by time: the past trains, the future tests. A random split would let the model peek at days next to the test days.

In [ ]:
n = len(X)
i_val, i_test = int(0.70 * n), int(0.85 * n)
X_train, y_train = X[:i_val], y[:i_val]
X_val, y_val = X[i_val:i_test], y[i_val:i_test]
X_test, y_test = X[i_test:], y[i_test:]
print("train", X_train.shape, " val", X_val.shape, " test", X_test.shape)

## 5. Scale, with the training numbers only

In [ ]:
mean, std = X_train.mean(axis=0), X_train.std(axis=0)       # from TRAIN only
y_mean, y_std = y_train.mean(), y_train.std()
scale_x = lambda a: (a - mean) / std
scale_y = lambda a: (a - y_mean) / y_std
unscale_y = lambda a: a * y_std + y_mean
print("train inputs after scaling: mean", scale_x(X_train).mean(0).round(2), " std", scale_x(X_train).std(0).round(2))

## 6. Feed: tensors and batches

A model eats **tensors** (arrays PyTorch understands), a few rows at a time (**a batch**).

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
to_t = lambda a: torch.tensor(a, dtype=torch.float32)
train_dl = DataLoader(TensorDataset(to_t(scale_x(X_train)), to_t(scale_y(y_train))), batch_size=32, shuffle=True)
Xv, yv = to_t(scale_x(X_val)), to_t(scale_y(y_val))

xb, yb = next(iter(train_dl))
print("one batch:", xb.shape, "→", yb.shape)     # 32 examples × 4 inputs → 32 × 1 output

## 7. The model: a function with knobs

Input size 4 (our 4 columns), output size 1 (one number). In between: one hidden layer.

In [ ]:
def make_model(hidden=16):
    return nn.Sequential(nn.Linear(4, hidden), nn.ReLU(), nn.Linear(hidden, 1))

model = make_model()
print(model)
print("parameters (knobs):", sum(p.numel() for p in model.parameters()))   # 4·16+16 + 16·1+1 = 97

## 8. Train: the loop every model uses

For each batch: **forward** (guess), **loss** (how wrong), **backward** (which way to turn each knob), **step** (turn them). After each pass over the data (an **epoch**), check the validation loss and keep the best model.

In [ ]:
import copy

def train(model, lr=1e-2, epochs=60):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            pred = model(xb)               # forward
            loss = loss_fn(pred, yb)       # loss
            opt.zero_grad()
            loss.backward()                # backward: gradients
            opt.step()                     # update the knobs
        model.eval()
        with torch.no_grad():
            val = loss_fn(model(Xv), yv).item()   # validation, no learning
        history.append(val)
        if val < best:
            best, best_state = val, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)      # keep the best epoch
    return best, history

best, history = train(model)
print("best validation loss:", round(best, 4), " at epoch", int(np.argmin(history)))

## 9. Size: parameters against examples

A rough check, not a law: far more knobs than examples means the model can learn the examples by heart.

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params} parameters for {len(X_train)} training examples: {n_params / len(X_train):.2f} per example")

## 10. Settings (hyperparameters): chosen on validation, never on test

The **parameters** (weights) are learned. The **settings** are chosen by us: layer size, learning rate, number of epochs, batch size. Try a few, keep the one with the lowest *validation* loss.

In [ ]:
results = []
for hidden in [4, 16, 64]:
    for lr in [1e-3, 1e-2]:
        torch.manual_seed(0)
        m = make_model(hidden)
        v, _ = train(m, lr=lr)
        results.append((hidden, lr, sum(p.numel() for p in m.parameters()), round(v, 4)))
pd.DataFrame(results, columns=["hidden", "lr", "parameters", "validation loss"]).sort_values("validation loss")

In [ ]:
hidden, lr = sorted(results, key=lambda r: r[3])[0][:2]    # the best settings
torch.manual_seed(0)
model = make_model(hidden)
train(model, lr=lr)
print("chosen: hidden =", hidden, " lr =", lr)

## 11. Test: once, in real units, against a baseline

**Baseline:** "tomorrow = today". If the model can't beat that, it is not worth having.

In [ ]:
model.eval()
with torch.no_grad():
    pred = unscale_y(model(to_t(scale_x(X_test))).numpy())
mae_model = np.abs(pred - y_test).mean()
mae_baseline = np.abs(X_test[:, 3:4] - y_test).mean()     # demand_today as the guess
print(f"test error, model:    {mae_model:6.1f} thousand litres per day")
print(f"test error, baseline: {mae_baseline:6.1f} thousand litres per day")

## 12. Save everything needed to use the model again

In [ ]:
torch.save({"weights": model.state_dict(), "hidden": hidden,
            "mean": mean, "std": std, "y_mean": y_mean, "y_std": y_std,
            "inputs": list(data.columns[:-1])}, "demand_model.pt")

ck = torch.load("demand_model.pt", weights_only=False)        # later, somewhere else
m2 = make_model(ck["hidden"]); m2.load_state_dict(ck["weights"]); m2.eval()
x_new = np.array([[31.0, 0.0, 0, 1450.0]], dtype=np.float32)  # hot, dry weekday, 1450 today
with torch.no_grad():
    print("forecast for tomorrow:", float((m2(to_t((x_new - ck["mean"]) / ck["std"])) * ck["y_std"] + ck["y_mean"])[0, 0]))

**That is the whole skeleton.** Notebooks 1 and 2 do the same twelve steps on the real plant data: a messier file, a time series instead of a table, and a harder question.